## Stage 7 - ZEN

### Dataset Exploration

In [1]:
import os

cholec = "Cholec80/cholec80"

# checking videos
videos = sorted(os.listdir(f"{cholec}/videos"))
print(f"Videos: {len(videos)}")
print(f"Sample : {videos[:3]}")


# checking phase annotation format
phase_files = sorted(os.listdir(f'{cholec}/phase_annotations'))
print(f"\nPhase annotation files: {len(phase_files)}")
print(f"First few: {phase_files[:3]}")

# reading one annotation file to check
with open(f"{cholec}/phase_annotations/{phase_files[0]}") as f:
    lines = f.readlines()
print(f"\nFirst annotation file: {phase_files[0]}")
print(f"Total lines: {len(lines)}")
print(f"Header: {repr(lines[0])}")
print(f"Line 1: {repr(lines[1])}")
print(f"Line 100: {repr(lines[100])}")
print(f"Last line: {repr(lines[-1])}")

Videos: 160
Sample : ['video01-timestamp.txt', 'video01.mp4', 'video02-timestamp.txt']

Phase annotation files: 80
First few: ['video01-phase.txt', 'video02-phase.txt', 'video03-phase.txt']

First annotation file: video01-phase.txt
Total lines: 43327
Header: 'Frame\tPhase\n'
Line 1: '0\tPreparation\n'
Line 100: '99\tPreparation\n'
Last line: '43325\tGallbladderRetraction\n'


In [2]:
import os
import av

cholec = "Cholec80/cholec80"

# check one video loads
video_path = f"{cholec}/videos/video01.mp4"
container = av.open(video_path)
stream = container.streams.video[0]
print(f"Video01:")
print(f"  Duration: {float(stream.duration * stream.time_base):.1f}s")
print(f"  FPS: {float(stream.average_rate):.1f}")
print(f"  Resolution: {stream.width}x{stream.height}")
container.close()

# check annotation
ann_path = f"{cholec}/phase_annotations/video01-phase.txt"
with open(ann_path) as f:
    lines = f.readlines()[1:]
phases = [l.strip().split('\t')[1] for l in lines if '\t' in l]
from collections import Counter
counts = Counter(phases)
print(f"\nVideo01 phase distribution:")
for phase, count in sorted(counts.items()):
    print(f"  {phase}: {count} frames ({count/25:.0f}s)")

Video01:
  Duration: 1733.0s
  FPS: 25.0
  Resolution: 854x480

Video01 phase distribution:
  CalotTriangleDissection: 16300 frames (652s)
  CleaningCoagulation: 1825 frames (73s)
  ClippingCutting: 5350 frames (214s)
  GallbladderDissection: 14575 frames (583s)
  GallbladderPackaging: 2450 frames (98s)
  GallbladderRetraction: 2301 frames (92s)
  Preparation: 525 frames (21s)


### Imports and Phase map

In [3]:
import os
import av
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.metrics import f1_score
from collections import Counter
import time
import random


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

Device: mps


In [4]:
CHOLEC_ROOT = "Cholec80/cholec80"

PHASE_MAP = {
    'Preparation':              0,
    'CalotTriangleDissection':  1,
    'ClippingCutting':          2,
    'GallbladderDissection':    3,
    'GallbladderPackaging':     4,
    'CleaningCoagulation':      5,
    'GallbladderRetraction':    6
}
PHASE_NAMES = {v: k for k, v in PHASE_MAP.items()}
print(f"Phases: {len(PHASE_MAP)}")
print(f"Phase map: {PHASE_MAP}")

Phases: 7
Phase map: {'Preparation': 0, 'CalotTriangleDissection': 1, 'ClippingCutting': 2, 'GallbladderDissection': 3, 'GallbladderPackaging': 4, 'CleaningCoagulation': 5, 'GallbladderRetraction': 6}


### Cholec 80 Dataset

In [5]:
class Cholec80Dataset(Dataset):
    def __init__(self, video_ids, clip_len=8, sample_every=250):
        # video ids are list of ints like 1-40 for train 
        # clip len is frames per clip 
        # sample every is sample one clip every N frames - 250 in our case

        self.clip_len = clip_len
        self.clips = []     # (video_path, [frames_indices], label)

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((224, 224), antialias=True),
            transforms.Normalize([0.485,0.456,0.406],
                                  [0.229,0.224,0.225])
        ])

        for vid_id in video_ids:
            vid_str = f"video{vid_id:02d}"
            vid_path = f"{CHOLEC_ROOT}/videos/{vid_str}.mp4"
            ann_path = f"{CHOLEC_ROOT}/phase_annotations/{vid_str}-phase.txt"

            if not os.path.exists(vid_path):
                continue

            # parse frame-level labels

            frame_labels = {}

            with open(ann_path) as f:
                for line in f.readlines()[1:]:
                    parts = line.strip().split('\t')
                    if len(parts) == 2 and parts[1] in PHASE_MAP:
                        frame_labels[int(parts[0])] = PHASE_MAP[parts[1]]

            if not frame_labels:
                continue

            # to know when to stop sampling
            max_frame = max(frame_labels.keys())

            # sample clips
            for start in range(0, max_frame - clip_len * 2, sample_every):
                # get clip_len evenly spaced frame indices
                frame_ids = list(np.linspace(
                    start, start+clip_len-1,
                    clip_len, dtype=int
                ))
                center = frame_ids[clip_len // 2]
                if center not in frame_labels:
                    continue
                
                label = frame_labels[center]
                self.clips.append((vid_path, start, label))

        print(f"Videos {video_ids[0]:02d}–{video_ids[-1]:02d}: "
              f"{len(self.clips)} clips")
        
    def _load_frames(self, video_path, start_frame):
        # load frames

        frames    = []
        container = av.open(video_path)
        stream    = container.streams.video[0]

        # seek directly to start frame
        fps        = float(stream.average_rate)
        time_base  = float(stream.time_base)
        target_pts = int(start_frame / fps / time_base)
        container.seek(target_pts, stream=stream)

        # decode only clip_len consecutive frames
        for frame in container.decode(video=0):
            img = frame.to_ndarray(format='rgb24')
            frames.append(self.transform(img))
            if len(frames) == self.clip_len:
                break
        container.close()

        while len(frames) < self.clip_len:
            frames.append(frames[-1])
        return torch.stack(frames)  # (T, 3, 224, 224)
    
    def __len__(self): 
        return len(self.clips)

    def __getitem__(self, idx):
        vid_path, start_frame, label = self.clips[idx]
        frames = self._load_frames(vid_path, start_frame)
        return frames, label
    

# standard Cholec80 split — videos 1-40 train, 41-80 test
print("Building datasets")
train_ids = list(range(1, 41))
test_ids  = list(range(41, 81))

train_ds = Cholec80Dataset(train_ids, sample_every=250)
test_ds  = Cholec80Dataset(test_ids,  sample_every=250)

# phase distribution
train_phases = Counter([c[2] for c in train_ds.clips])
print(f"\nTrain phase distribution:")
for pid in sorted(train_phases):
    print(f"  {PHASE_NAMES[pid]}: {train_phases[pid]} clips")
print(f"\nTotal train: {len(train_ds)} | Total test: {len(test_ds)}")

Building datasets
Videos 01–40: 8647 clips
Videos 41–80: 9841 clips

Train phase distribution:
  Preparation: 390 clips
  CalotTriangleDissection: 3691 clips
  ClippingCutting: 732 clips
  GallbladderDissection: 2414 clips
  GallbladderPackaging: 370 clips
  CleaningCoagulation: 726 clips
  GallbladderRetraction: 324 clips

Total train: 8647 | Total test: 9841


In [11]:
# cap test set to 500 clips — faster evaluation, still representative
g_test = torch.Generator()
g_test.manual_seed(SEED)
test_indices    = torch.randperm(len(test_ds), generator=g_test)[:500].tolist()
test_ds_small   = Subset(test_ds, test_indices)
test_loader     = DataLoader(test_ds_small, batch_size=8,
                              shuffle=False, num_workers=0)
print(f"Test subset: {len(test_ds_small)} clips (~{500*0.1/60:.1f} min per eval)")

Test subset: 500 clips (~0.8 min per eval)


### Phase model

In [7]:
class DinoPhaseModel(nn.Module):
    # DINO-ViT + Temporal Transformer for phase recognition
    # Same architecture as Stage 3 DinoSurgicalTransformer but with linear classifier instead of prototypes
    # Can be initialized randomly OR from Stage 3 checkpoint

    def __init__(self, n_classes=7, clip_len=8, feat_dim=384,
                 n_heads = 6, n_layers=4, dropout=0.1):
        super().__init__()
        self.clip_len = clip_len

        # frozen DINO-Vit 
        self.vit = timm.create_model(
            'vit_small_patch16_224.dino',
            pretrained=True, num_classes=0
        )
        for p in self.vit.parameters():
            p.requires_grad = False

        # temporal transformer randomly init here - can be overwritten by other pretrained weights
        self.cls_token = nn.Parameter(torch.randn(1, 1, feat_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, clip_len+1, feat_dim))
        encoder_layer  = nn.TransformerEncoderLayer(
            d_model=feat_dim, nhead=n_heads,
            dim_feedforward=feat_dim*4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.norm        = nn.LayerNorm(feat_dim)

        self.classifier = nn.Linear(feat_dim, n_classes)

    def forward(self, x):
        # (B, T, 3, H, W) -> (B, n_classes)
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)
        with torch.no_grad():
            x = self.vit(x)  # (B*T, 384)
        x = x.view(B, T, -1)   # (B, T, 384)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = x + self.pos_embed
        x   = self.transformer(x)
        x   = self.norm(x)
        return self.classifier(x[:, 0, :])  # (B, n_classes)
    
torch.manual_seed(SEED)
test_model = DinoPhaseModel(n_classes=7, clip_len=8).to(device)
total = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"Trainable params: {total:,}  (ViT frozen)")
print(f"Device: {device}")
del test_model

Trainable params: 7,105,159  (ViT frozen)
Device: mps


### Training

In [8]:
from tqdm.notebook import tqdm as tqdm_nb

def train_phase_model(model, train_loader, test_loader,
                      n_epochs=10, label=''):
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=1e-6
    )
    criterion = nn.CrossEntropyLoss()
    best_f1   = 0

    print(f"\n{label}")
    print("="*50)

    for epoch in tqdm_nb(range(1, n_epochs+1), desc="Training"):
        t0 = time.time()

        # train
        model.train()
        tr_preds, tr_labels = [], []
        for clips, labels in train_loader:
            clips, labels = clips.to(device), labels.to(device)
            logits = model(clips)
            loss   = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
            optimizer.step()
            tr_preds.extend(logits.argmax(1).cpu().numpy())
            tr_labels.extend(labels.cpu().numpy())
        scheduler.step()
        tr_f1 = f1_score(tr_labels, tr_preds,
                          average='macro', zero_division=0)

        # eval
        model.eval()
        te_preds, te_labels = [], []
        with torch.no_grad():
            for clips, labels in test_loader:
                clips, labels = clips.to(device), labels.to(device)
                logits = model(clips)
                te_preds.extend(logits.argmax(1).cpu().numpy())
                te_labels.extend(labels.cpu().numpy())
        te_f1 = f1_score(te_labels, te_preds,
                          average='macro', zero_division=0)

        if te_f1 > best_f1:
            best_f1 = te_f1

        tqdm_nb.write(f"Epoch {epoch:2d} | Train F1: {tr_f1:.3f} | "
                      f"Test F1: {te_f1:.3f} | {time.time()-t0:.1f}s")

    print(f"Best Test F1: {best_f1:.3f}")
    return best_f1

### Loading stage 3 checkpoint

In [9]:
def load_stage3_weights(model, checkpoint_path="stage3_combined_best.pt"):
    # load stage 3 temporal transformer weights onto DinoPhaseModel
    # skip layers that dont match i.e. proto and proj head
    # returns number of matched layers

    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint not found: {checkpoint_path}")
        return 0
    
    ckpt = torch.load(checkpoint_path, map_location=device)
    pretrained = ckpt['model_state_dict']
    model_dict = model.state_dict()

    # match by key name and shape
    matched = {
        k: v for k, v in pretrained.items()
        if k in model_dict and v.shape == model_dict[k].shape
    }
    model_dict.update(matched)
    model.load_state_dict(model_dict)

    print(f"Loaded: {checkpoint_path}")
    print(f"Matched layers: {len(matched)}/{len(model_dict)}")
    print(f"Pretrained F1 was: {ckpt.get('best_f1', 'N/A')}")
    return len(matched)

# testing it
torch.manual_seed(SEED)
test_m = DinoPhaseModel(n_classes=7).to(device)
matched = load_stage3_weights(test_m)
del test_m


Loaded: stage3_combined_best.pt
Matched layers: 202/204
Pretrained F1 was: 0.7909992093595457


### Running the experiment

In [10]:
print("Stage 7 - ZEN Concept: Label Efficiency on Cholec80")
print("="*50)
print("Comparing: ")
print("     A) Random init Temporal Transformer (baseline)")
print("     B) Stage 3 pretrained Temporal Transformer (foundation)")

fractions = [0.10, 0.25, 0.50, 1.00]
scratch_f1s = {}
pretrain_f1s = {}

# test set is the capped 2000 test set for all runs

for frac in fractions:
    n = int(len(train_ds)*frac)
    print(f"\nTraining data: {frac*100:.0f}% = {n} clips")

    # reproducible subset
    torch.manual_seed(SEED)
    indices = torch.randperm(len(train_ds))[:n].tolist()
    subset  = Subset(train_ds, indices)
    g2 = torch.Generator()
    g2.manual_seed(SEED)
    loader = DataLoader(subset, batch_size=8,
                        shuffle=True, num_workers=0, generator=g2)
    
    # from scratch model
    torch.manual_seed(SEED)
    model_a = DinoPhaseModel(n_classes=7, clip_len=8).to(device)
    f1_a = train_phase_model(
        model_a, loader, test_loader,
        n_epochs=8,
        label=f"A) Scratch — {frac*100:.0f}% data"
    )
    scratch_f1s[frac] = f1_a
    del model_a

    # stage 3 pretrained model
    torch.manual_seed(SEED)
    model_b = DinoPhaseModel(n_classes=7, clip_len=8).to(device)
    load_stage3_weights(model_b)
    f1_b = train_phase_model(
        model_b, loader, test_loader,
        n_epochs=8,
        label=f"B) Pretrained — {frac*100:.0f}% data"
    )
    pretrain_f1s[frac] = f1_b 
    del model_b

# final results table

print("="*50)
print(f"  {'Data':>6} | {'Scratch':>8} | {'Pretrained':>10} | {'Gain':>8}")
print(f"  {'-'*50}")
for frac in fractions:
    s    = scratch_f1s[frac]
    p    = pretrain_f1s[frac]
    gain = f"+{p-s:.3f}" if p >= s else f"{p-s:.3f}"
    print(f"  {frac*100:>5.0f}% | {s:>8.3f} | {p:>10.3f} | {gain:>8}")

Stage 7 - ZEN Concept: Label Efficiency on Cholec80
Comparing: 
     A) Random init Temporal Transformer (baseline)
     B) Stage 3 pretrained Temporal Transformer (foundation)

Training data: 10% = 864 clips

A) Scratch — 10% data


Training:   0%|          | 0/8 [00:00<?, ?it/s]

Epoch  1 | Train F1: 0.329 | Test F1: 0.320 | 210.0s
Epoch  2 | Train F1: 0.631 | Test F1: 0.493 | 209.3s
Epoch  3 | Train F1: 0.778 | Test F1: 0.493 | 205.2s
Epoch  4 | Train F1: 0.889 | Test F1: 0.475 | 206.3s
Epoch  5 | Train F1: 0.956 | Test F1: 0.519 | 206.0s
Epoch  6 | Train F1: 0.990 | Test F1: 0.521 | 206.2s


KeyboardInterrupt: 